In [ ]:
import pandas as pd
import numpy as np

DATA = '../data/individual/processed'
PSY = '../data/individual/psychometric'

def calculate_hrv_metrics(ibi_data):
    valid_ibi = pd.Series(ibi_data)
    diff_nn = np.diff(valid_ibi)
    rmssd = np.sqrt(np.mean(np.square(diff_nn)))
    sdnn = np.std(valid_ibi, ddof=1)
    return rmssd, sdnn

ibi_baseline = pd.read_csv(f'{DATA}/ibi.csv')
ibi_01 = pd.read_csv(f'{DATA}/ibi_01.csv')
ibi_02 = pd.read_csv(f'{DATA}/ibi_02.csv')
ibi_03 = pd.read_csv(f'{DATA}/ibi_03.csv')

# physiological bounds
ibi_baseline = ibi_baseline[(ibi_baseline['ibi'] > 300) & (ibi_baseline['ibi'] < 2000)]
rmssd_baseline, sdnn_baseline = calculate_hrv_metrics(ibi_baseline['ibi'])
baseline_metrics = (rmssd_baseline, sdnn_baseline)

psychometric_01 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_01.csv')
psychometric_02 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_02.csv')
psychometric_03 = pd.read_csv(f'{PSY}/Psychometric_Test_Results_03.csv')

psychometric_01['Question Start Time'] = pd.to_datetime(psychometric_01['Question Start Time'], utc=True, errors='coerce').dt.tz_convert(None)
psychometric_01['Question Answer Time'] = pd.to_datetime(psychometric_01['Question Answer Time'], utc=True, errors='coerce').dt.tz_convert(None)
psychometric_02['Question Start Time'] = pd.to_datetime(psychometric_02['Question Start Time'], utc=True, errors='coerce').dt.tz_convert(None)
psychometric_02['Question Answer Time'] = pd.to_datetime(psychometric_02['Question Answer Time'], utc=True, errors='coerce').dt.tz_convert(None)
psychometric_03['Question Start Time'] = pd.to_datetime(psychometric_03['Question Start Time'], utc=True, errors='coerce').dt.tz_convert(None)
psychometric_03['Question Answer Time'] = pd.to_datetime(psychometric_03['Question Answer Time'], utc=True, errors='coerce').dt.tz_convert(None)

psychometric_01 = psychometric_01.dropna(subset=['Question Start Time'])
psychometric_02 = psychometric_02.dropna(subset=['Question Start Time'])
psychometric_03 = psychometric_03.dropna(subset=['Question Start Time'])

types_count = {'HADS': 14, 'STAI-S': 20, 'STAI-T': 20, 'BFI': 10, 'FQ': 24}

def filter_correct_questions(df, types_count):
    filtered_df = pd.DataFrame()
    for q_type, count in types_count.items():
        filtered_df = pd.concat([filtered_df, df[df['Type'] == q_type].head(count)])
    return filtered_df

questions_01 = filter_correct_questions(psychometric_01, types_count)
questions_02 = filter_correct_questions(psychometric_02, types_count)
questions_03 = filter_correct_questions(psychometric_03, types_count)

def prepare_ibi(ibi_data):
    # convert once, avoid repeated mutation
    ibi = ibi_data.copy()
    ibi['datetime'] = pd.to_datetime(ibi['datetime'], utc=True, errors='coerce').dt.tz_convert(None)
    return ibi

ibi_01 = prepare_ibi(ibi_01)
ibi_02 = prepare_ibi(ibi_02)
ibi_03 = prepare_ibi(ibi_03)

def get_ibi_data(question, ibi_data):
    filtered = ibi_data[
        (ibi_data['datetime'] >= question['Question Start Time']) &
        (ibi_data['datetime'] <= question['Question Answer Time'])
    ]
    return filtered['ibi'].values

def detect_significant_decrease(test_metrics, baseline_metrics):
    return (
        test_metrics[0] < baseline_metrics[0],
        test_metrics[1] < baseline_metrics[1]
    )

def calculate_hrv_metrics_for_questions(questions, ibi_data, baseline_metrics):
    results = []
    for _, question in questions.iterrows():
        ibi_values = get_ibi_data(question, ibi_data)
        # physiological bounds
        ibi_values = ibi_values[(ibi_values > 300) & (ibi_values < 2000)]
        if len(ibi_values) > 1:
            rmssd, sdnn = calculate_hrv_metrics(ibi_values)
            sig = detect_significant_decrease((rmssd, sdnn), baseline_metrics)
            results.append({
                'Type': question['Type'],
                'Start Time': question['Question Start Time'],
                'End Time': question['Question Answer Time'],
                'Score': question['Answer'],
                'RMSSD': round(rmssd, 2),
                'SDNN': round(sdnn, 2),
                'RMSSD Decrease': 'Yes' if sig[0] else 'No',
                'SDNN Decrease': 'Yes' if sig[1] else 'No'
            })
    return pd.DataFrame(results)

results_01 = calculate_hrv_metrics_for_questions(questions_01, ibi_01, baseline_metrics)
results_02 = calculate_hrv_metrics_for_questions(questions_02, ibi_02, baseline_metrics)
results_03 = calculate_hrv_metrics_for_questions(questions_03, ibi_03, baseline_metrics)

results_01['Test'] = 'Test 01'
results_02['Test'] = 'Test 02'
results_03['Test'] = 'Test 03'

combined_results = pd.concat([results_01, results_02, results_03], ignore_index=True)
combined_results = combined_results[['Test', 'Type', 'Start Time', 'End Time', 'Score', 'RMSSD', 'SDNN', 'RMSSD Decrease', 'SDNN Decrease']]
combined_results.to_csv(f'{DATA}/QQHRV.csv', index=False)

combined_results.head()
